# Aula 5 — KNN: classificação por vizinhos mais próximos

**Trilha de IA Aplicada — Projeto Cuidadores**  
**Notebook do estudante**

Nesta aula, construiremos um classificador para estimar se uma situação apresenta **menor ou maior possibilidade de atraso**, representada didaticamente pelas classes:

- `0` → não atrasou;
- `1` → atrasou.

> **Atenção:** os dados são sintéticos e a atividade é exclusivamente educacional. O modelo não está validado para decisões reais de saúde.


## Objetivos de aprendizagem

Ao final da prática, você deverá conseguir:

1. identificar features e target;
2. separar dados de treino e teste;
3. padronizar features com `StandardScaler`;
4. treinar um modelo KNN;
5. testar diferentes valores de K;
6. calcular e interpretar a acurácia;
7. comparar KNN e Árvore de Decisão;
8. preparar novos registros antes de usar `predict()`.


## 1. Preparando o ambiente

Execute a célula abaixo para importar as bibliotecas.


In [ ]:
import io
import os

import matplotlib.pyplot as plt
import pandas as pd

from IPython.display import display
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

print("Ambiente preparado!")


## 2. Carregando o dataset principal

O notebook procura o arquivo `atrasos_medicamentos.csv`. Se ele ainda não estiver no ambiente do Colab, aparecerá uma janela para upload.


In [ ]:
def carregar_csv(nome_arquivo):
    """Carrega um CSV local ou solicita upload quando estiver no Google Colab."""
    if os.path.exists(nome_arquivo):
        print(f"Arquivo encontrado: {nome_arquivo}")
        return pd.read_csv(nome_arquivo)

    try:
        from google.colab import files

        print(f"Selecione o arquivo: {nome_arquivo}")
        enviados = files.upload()

        if nome_arquivo not in enviados:
            raise FileNotFoundError(
                f"O arquivo enviado não se chama {nome_arquivo}. "
                "Renomeie-o ou ajuste a variável no notebook."
            )

        return pd.read_csv(io.BytesIO(enviados[nome_arquivo]))
    except ImportError as erro:
        raise FileNotFoundError(
            f"Coloque o arquivo {nome_arquivo} na mesma pasta do notebook."
        ) from erro


dados = carregar_csv("atrasos_medicamentos.csv")
print("Dimensões do dataset:", dados.shape)
display(dados.head())


## 3. Conhecendo os dados

Antes de treinar qualquer modelo, devemos verificar estrutura, tipos, valores ausentes, duplicidades e estatísticas básicas.


In [ ]:
dados.info()


In [ ]:
print("Valores ausentes por coluna:")
display(dados.isnull().sum().to_frame("quantidade"))

print("Registros duplicados:", dados.duplicated().sum())


In [ ]:
display(dados.describe())


In [ ]:
distribuicao = (
    dados["medicamento_atrasado"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Não atrasou (0)", 1: "Atrasou (1)"})
)

display(distribuicao.to_frame("quantidade"))


### Observe e responda

1. Quantas linhas e colunas existem?
2. Há valores ausentes?
3. A coluna `medicamento_atrasado` possui quais classes?
4. As duas classes aparecem em quantidade semelhante?


## 4. Definindo features e target

Nesta primeira experiência, usaremos somente duas features numéricas:

- `quantidade_lembretes`;
- `atraso_medio`.

O target será `medicamento_atrasado`.


In [ ]:
features = ["quantidade_lembretes", "atraso_medio"]
target = "medicamento_atrasado"

X = dados[features]
y = dados[target]

print("Features (X):")
display(X.head())

print("Target (y):")
display(y.head().to_frame())


## 5. Separando treino e teste

- **Treino:** exemplos utilizados pelo modelo para aprender.
- **Teste:** exemplos separados para avaliar o comportamento do modelo.


In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Linhas de treino:", len(X_treino))
print("Linhas de teste:", len(X_teste))


## 6. Padronizando as features

O KNN usa distâncias. Por isso, features com escalas diferentes podem influenciar o cálculo de forma desigual.

No treino usamos `fit_transform()`; no teste usamos somente `transform()`.


In [ ]:
scaler = StandardScaler()

X_treino_escalado = scaler.fit_transform(X_treino)
X_teste_escalado = scaler.transform(X_teste)

print("Médias aprendidas pelo scaler:", scaler.mean_.round(2))
print("Desvios-padrão aprendidos:", scaler.scale_.round(2))


In [ ]:
treino_escalado_df = pd.DataFrame(
    X_treino_escalado,
    columns=features,
    index=X_treino.index
)

display(treino_escalado_df.describe().round(2))


## 7. Criando e treinando o KNN com K = 3

`n_neighbors=3` significa que o algoritmo observará os três vizinhos mais próximos.


In [ ]:
modelo_knn = KNeighborsClassifier(n_neighbors=3)

modelo_knn.fit(X_treino_escalado, y_treino)

print("Modelo KNN treinado com K = 3!")


## 8. Fazendo previsões e calculando a acurácia


In [ ]:
previsoes_knn = modelo_knn.predict(X_teste_escalado)

acuracia_knn = accuracy_score(y_teste, previsoes_knn)

print(f"Acurácia do KNN com K = 3: {acuracia_knn:.2%}")


In [ ]:
comparacao_previsoes = pd.DataFrame({
    "valor_real": y_teste.values,
    "previsao_knn": previsoes_knn
})

display(comparacao_previsoes.head(10))


### Interpretação

A acurácia informa a proporção de acertos **neste conjunto de teste**. Ela não prova que o modelo está pronto para uso real.


## 9. Testando diferentes valores de K

Vamos testar K = 1, 3, 5, 7 e 9 mantendo os mesmos dados de treino e teste.


In [ ]:
resultados_knn = []

for k in [1, 3, 5, 7, 9]:
    modelo = KNeighborsClassifier(n_neighbors=k)
    modelo.fit(X_treino_escalado, y_treino)
    previsoes = modelo.predict(X_teste_escalado)
    acuracia = accuracy_score(y_teste, previsoes)

    resultados_knn.append({
        "modelo": "KNN",
        "configuracao": f"K = {k}",
        "k": k,
        "acuracia": acuracia
    })

resultados_knn_df = pd.DataFrame(resultados_knn)
display(resultados_knn_df[["modelo", "configuracao", "acuracia"]])


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    resultados_knn_df["k"],
    resultados_knn_df["acuracia"],
    marker="o"
)
plt.xticks(resultados_knn_df["k"])
plt.ylim(0, 1.05)
plt.xlabel("Valor de K")
plt.ylabel("Acurácia")
plt.title("Acurácia do KNN para diferentes valores de K")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
melhor_linha = resultados_knn_df.loc[resultados_knn_df["acuracia"].idxmax()]

print("Melhor resultado do experimento:")
print("Configuração:", melhor_linha["configuracao"])
print(f"Acurácia: {melhor_linha['acuracia']:.2%}")


### Analise

1. O resultado mudou quando K foi alterado?
2. Qual K apresentou a maior acurácia?
3. Podemos afirmar que esse K será sempre o melhor? Justifique.


## 10. Comparando com a Árvore de Decisão

A Árvore de Decisão será treinada com as features originais, usando os mesmos conjuntos de treino e teste.


In [ ]:
arvore = DecisionTreeClassifier(random_state=RANDOM_STATE)

arvore.fit(X_treino, y_treino)
previsoes_arvore = arvore.predict(X_teste)

acuracia_arvore = accuracy_score(y_teste, previsoes_arvore)

print(f"Acurácia da Árvore de Decisão: {acuracia_arvore:.2%}")


In [ ]:
comparacao_modelos = pd.concat([
    resultados_knn_df[["modelo", "configuracao", "acuracia"]],
    pd.DataFrame([{
        "modelo": "Árvore de Decisão",
        "configuracao": "padrão da atividade",
        "acuracia": acuracia_arvore
    }])
], ignore_index=True)

comparacao_modelos = comparacao_modelos.sort_values(
    "acuracia",
    ascending=False
).reset_index(drop=True)

comparacao_formatada = comparacao_modelos.copy()
comparacao_formatada["acuracia"] = comparacao_formatada["acuracia"].map(
    lambda valor: f"{valor:.2%}"
)

display(comparacao_formatada)


### Compare os algoritmos

1. Qual configuração apresentou maior acurácia neste experimento?
2. A Árvore de Decisão precisou do `StandardScaler` nesta atividade?
3. Qual é a diferença conceitual entre KNN e Árvore de Decisão?
4. Por que não podemos escolher um algoritmo apenas pelo nome?


## 11. Fazendo uma nova previsão

A nova entrada deve possuir as mesmas features e passar pelo mesmo scaler usado no treino.


In [ ]:
novo_registro = pd.DataFrame(
    [[3, 20]],
    columns=features
)

novo_registro_escalado = scaler.transform(novo_registro)
resultado = modelo_knn.predict(novo_registro_escalado)[0]

rotulo = "ATRASOU" if resultado == 1 else "NÃO ATRASOU"

print("Novo registro:")
display(novo_registro)
print("Previsão:", resultado, "→", rotulo)


## 12. Classificando vários registros novos

Agora usaremos o segundo dataset: `novos_registros_medicamentos.csv`.


In [ ]:
novos = carregar_csv("novos_registros_medicamentos.csv")

print("Novos registros:")
display(novos)


In [ ]:
X_novos = novos[features]
X_novos_escalado = scaler.transform(X_novos)

previsoes_novos = modelo_knn.predict(X_novos_escalado)

resultado_novos = novos.copy()
resultado_novos["previsao"] = previsoes_novos
resultado_novos["previsao_texto"] = resultado_novos["previsao"].map({
    0: "Não atrasou",
    1: "Atrasou"
})

display(resultado_novos)


In [ ]:
nome_saida = "previsoes_novos_registros.csv"
resultado_novos.to_csv(nome_saida, index=False)

print(f"Arquivo gerado: {nome_saida}")

# No Google Colab, retire o # das duas linhas abaixo para baixar o resultado:
# from google.colab import files
# files.download(nome_saida)


## 13. Desafio do estudante

Faça uma nova experiência:

1. escolha um valor de K que ainda não foi testado;
2. treine o modelo;
3. calcule a acurácia;
4. compare com os resultados anteriores;
5. escreva uma conclusão.


In [ ]:
# SUA VEZ

k_escolhido = 11  # altere este valor

modelo_desafio = KNeighborsClassifier(n_neighbors=k_escolhido)
modelo_desafio.fit(X_treino_escalado, y_treino)

previsoes_desafio = modelo_desafio.predict(X_teste_escalado)
acuracia_desafio = accuracy_score(y_teste, previsoes_desafio)

print("K escolhido:", k_escolhido)
print(f"Acurácia: {acuracia_desafio:.2%}")


## 14. Conclusão do grupo

Escreva pelo menos cinco linhas respondendo:

- O que o valor K representa?
- Por que a padronização foi necessária?
- Qual configuração apresentou maior acurácia?
- Qual foi a diferença observada entre KNN e Árvore de Decisão?
- Por que estes resultados não validam o modelo para uso real em saúde?

**Conclusão:**  
_Escreva aqui..._


## Resumo

Nesta prática, percorremos o fluxo:

**dados → features e target → treino e teste → padronização → KNN → previsão → acurácia → comparação**

A ideia principal é:

> O KNN classifica um novo exemplo observando as classes dos exemplos conhecidos que estão mais próximos dele.
